In [42]:
# imports
using JuMP
using SCIP
import MathOptInterface as MOI
using PrettyTables
include("util.jl")

generate_weights_spherical (generic function with 1 method)

In [43]:
# Параметры модели Банистера
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7
α = 0.5

0.5

In [44]:
# Файл с входными данными
input_files_folder = "input"
# input_file_name = "input_smaller.txt"
input_file_name = "input_current.txt"

file_path = input_files_folder * "/" * input_file_name

"input/input_current.txt"

In [45]:
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

(56, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  47, 48, 49, 50, 51, 52, 53, 54, 55, 56], [6, 13, 20, 27, 34, 41, 48, 55], 18, SubString{String}["Player1", "Player2", "Player3", "Player4", "Player5", "Player6", "Player7", "Player8", "Player9", "Player10", "Player11", "Player12", "Player13", "Player14", "Player15", "Player16", "Player17", "Player18"], 38, SubString{String}["Ускорение", "Торможение", "Реакция", "ЛовляНаУрТуловища", "СменаХвата", "ПриставнойШаг", "СменаНаправления", "Лестница", "ПрофилактическиеУпражнения", "БеговыеУпражнения"  …  "Хаммер5Слабый", "Блейд5Сильный", "Хаммер10Сильный", "Блейд10Сильный", "Скубер10Сильный", "ФорхендБэкхенд25Вышагивание", "ФорхендБэкхенд50", "ХаммерБлейдСкубер25", "ПеревернутыйБэкхенд25", "ОбводнойБэкхенд25"], 10, SubString{String}["Зоны", "Роли", "ВертикальныйСтек", "ЛичнаяЗащита", "ГоризонтальныйСтек", "ДиагональныйСтек", "РазделенныйСтек", "СменаИПодстраховка", "АтакаВблизиГола", "СтандартныеПозиционныеСхемы"], 48, SubString{String}["Ускорение_1", "Т

In [46]:
model, x, y, z, Z, A, A_avg, B, C = setup_model(file_path)

(A JuMP Model
├ solver: SCIP
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 9688
├ num_constraints: 16872
│ ├ AffExpr in MOI.GreaterThan{Float64}: 6992
│ ├ AffExpr in MOI.LessThan{Float64}: 200
│ └ VariableRef in MOI.ZeroOne: 9680
└ Names registered in the model
  └ :A_min, :Z, :x, :y, :z, VariableRef[x[1,1] x[1,2] … x[1,55] x[1,56]; x[2,1] x[2,2] … x[2,55] x[2,56]; … ; x[47,1] x[47,2] … x[47,55] x[47,56]; x[48,1] x[48,2] … x[48,55] x[48,56]], VariableRef[y[1,1,1] y[1,2,1] … y[1,17,1] y[1,18,1]; y[2,1,1] y[2,2,1] … y[2,17,1] y[2,18,1]; … ; y[37,1,1] y[37,2,1] … y[37,17,1] y[37,18,1]; y[38,1,1] y[38,2,1] … y[38,17,1] y[38,18,1];;; y[1,1,2] y[1,2,2] … y[1,17,2] y[1,18,2]; y[2,1,2] y[2,2,2] … y[2,17,2] y[2,18,2]; … ; y[37,1,2] y[37,2,2] … y[37,17,2] y[37,18,2]; y[38,1,2] y[38,2,2] … y[38,17,2] y[38,18,2];;; y[1,1,3] y[1,2,3] … y[1,17,3] y[1,18,3]; y[2,1,3] y[2,2,3] … y[2,17,3] y[2,18,3]; … ; y[37,1,3] y[37,2,3] … y[37,17,3] y[37,18,3]; y[38,1,3] y[38,2,3] … y[38,17,3] y[38,18,3];;;

In [47]:
# Вычисление идеальной точки, определение функций нормировки
# ideal_point = compute_ideal_point(model)
# println("Ideal point: $ideal_point")

B_optimal = 398.0
ideal_point = [3654.73, 398.0, 20.0]

A_norm = () -> A()/ideal_point[1]
B_norm = () -> B()/ideal_point[2]
C_norm = () -> C()/ideal_point[3]

#450 (generic function with 1 method)

In [48]:
# Solve with all weight combinations via Local-MIP CLI

weights_for_moa = generate_weights_spherical()

function compute_A_from_vals(x_v)
    total = 0.0
    for d_idx in 1:length(D_star)
        d_day = D_star[d_idx]
        A_min_v = minimum(
            a0[p] + sum((δ1(D_[di], d_day) - δ2(D_[di], d_day)) * h[p][di] * sum(w[e]*x_v[e, di] for e in 1:E)
                   for di in 1:length(D_) if D_[di] <= d_day)
            for p in 1:P)
        A_avg_v = sum(
            a0[p] + sum((δ1(D_[di], d_day) - δ2(D_[di], d_day)) * h[p][di] * sum(w[e]*x_v[e, di] for e in 1:E)
                   for di in 1:length(D_) if D_[di] <= d_day)
            for p in 1:P) / P
        total += α * A_min_v + (1 - α) * A_avg_v
    end
    return total
end

output_folder = "results"
mkpath(output_folder)
solutions        = Vector{Vector{Float64}}()
solutions_nonrm  = Vector{Vector{Float64}}()
solution_weights = Vector{Vector{Float64}}()

weights_for_moa = [[1.0, 0.0, 0.0]]

# Решение задач с разными весами
for i in 1:length(weights_for_moa)
    # Свёртка целевой функции, запись задачи в .mps
        wA, wB, wC = weights_for_moa[i]
        @objective(model, Max, wA*A_norm() + wB*B_norm() + wC*C_norm())

        label = "[$(round(wA,digits=3)),$(round(wB,digits=3)),$(round(wC,digits=3))]"
        println("Solving combination $i/$(length(weights_for_moa)): $label")

        problem_name = "multi_$i"
        mps_path = problem_name * ".mps"
        sol_path  = problem_name * ".sol"
        write_to_file(model, mps_path)

    
    # Запуск решателя
        run(`./Local-MIP --model_file $mps_path --sol_path $sol_path --time_limit 30`)
        println("Solve complete")

    # Запись значений переменных решения и целевой функции в переменные Julia
        obj_val, var_vals = parse_localmip_solution(sol_path)

        if obj_val === nothing
            println("  No feasible solution found, skipping.\n")
            continue
        end

        x_vals = [round(Int, get(var_vals, "x[$e,$d_]", 0.0)) for e in 1:E, d_ in 1:length(D_)]
        y_vals = [round(Int, get(var_vals, "y[$s,$p,$d_]", 0.0)) for s in 1:S, p in 1:P, d_ in 1:length(D_star)]
        z_vals = [round(Int, get(var_vals, "z[$t,$p,$d_]", 0.0)) for t in 1:T, p in 1:P, d_ in 1:length(D_star)]
        Z_vals = [round(Int, get(var_vals, "Z[$t,$d_]",   0.0)) for t in 1:T, d_ in 1:length(D_star)]

        A_val  = compute_A_from_vals(x_vals)
        B_val  = Float64(sum(β(s, p, d_) * y_vals[s, p, d_] for s in 1:S, p in 1:P, d_ in 1:length(D_star)))
        C_val  = Float64(sum(γ(t, d_)    * Z_vals[t, d_]    for t in 1:T, d_ in 1:length(D_star)))
        An_val = A_val / ideal_point[1]
        Bn_val = B_val / ideal_point[2]
        Cn_val = C_val / ideal_point[3]

        push!(solutions,        [An_val, Bn_val, Cn_val])
        push!(solutions_nonrm,  [A_val,  B_val,  C_val])
        push!(solution_weights, [wA, wB, wC])

    # Запись результатов в файл
        output_file = "$(input_file_name)_$label"
        open(output_folder * "/" * output_file, "w") do io
            println(io, "Solution of model with weight distribution $label")
            println(io, "Ideal Point (Bound): ", ideal_point)

            println(io, "---Итоговая эффективность тренировок---")
            println(io, "A - общая физическая готовность команды\nB - бонус за освоенные индивидуальные навыки\nC - бонус за освоенные командой тактики\n")
            println(io, "A = $(A_val)\nB = $(B_val)\nC = $(C_val)\n\n")
            println(io, "---Итоговая эффективность тренировок относительно идеальной точки---")
            println(io, "A = $(An_val)\nB = $(Bn_val)\nC = $(Cn_val)\n\n")

            println(io, "---Расписание тренировок---\n")
            n_days = length(D_)
            data = Array{Any}(undef, n_days, 2)
            for d_ in 1:n_days
                data[d_, 1] = "Тренировка №$(d_) (день №$(D_[d_]))"
                day_exercises = String[]
                for e in 1:E
                    if x_vals[e, d_] >= 1; push!(day_exercises, Exercise_names[e]); end
                end
                data[d_, 2] = join(day_exercises, ", ")
            end
            pretty_table(io, data)

            println(io, "---Освоенные навыки---\n")
            for p in 1:P
                for d_ in 1:length(D_star)
                    println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
                    learned_anything = false
                    for s in 1:S
                        if y_vals[s, p, d_] == 1 && (d_ == 1 || y_vals[s, p, d_-1] != 1)
                            learned_anything = true
                            println(io, "Освоил навык '$(Skill_names[s])'")
                        end
                    end
                    if !learned_anything; println(io, "Ничего не освоил"); end
                end
                print(io, "\n")
            end

            println(io, "---Освоенные индивидуально тактики---\n")
            for p in 1:P
                for d_ in 1:length(D_star)
                    println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
                    learned_anything = false
                    for t in 1:T
                        if z_vals[t, p, d_] == 1 && (d_ == 1 || z_vals[t, p, d_-1] != 1)
                            learned_anything = true
                            println(io, "Освоил тактику '$(Tactic_names[t])'")
                        end
                    end
                    if !learned_anything; println(io, "Ничего не освоил"); end
                end
                print(io, "\n")
            end

            println(io, "---Освоенные командой тактики---\n")
            for d_ in 1:length(D_star)
                for t in 1:T
                    println(io, "Тактика: $(Tactic_names[t]), контрольная дата №$(d_) (день $(D_star[d_]))")
                    println(io, Z_vals[t, d_] == 1 ? "Освоено" : "Не освоено")
                end
                print(io, "\n")
            end
        end
        println("  Written: $(output_folder)/$(output_file)\n")
end

Weight combinations: 25
  1: A=1.0  B=0.0  C=0.0
  2: A=1.0  B=0.0  C=0.0
  3: A=1.0  B=0.0  C=0.0
  4: A=1.0  B=0.0  C=0.0
  5: A=1.0  B=0.0  C=0.0
  6: A=0.924  B=0.383  C=0.0
  7: A=0.924  B=0.354  C=0.146
  8: A=0.924  B=0.271  C=0.271
  9: A=0.924  B=0.146  C=0.354
  10: A=0.924  B=0.0  C=0.383
  11: A=0.707  B=0.707  C=0.0
  12: A=0.707  B=0.653  C=0.271
  13: A=0.707  B=0.5  C=0.5
  14: A=0.707  B=0.271  C=0.653
  15: A=0.707  B=0.0  C=0.707
  16: A=0.383  B=0.924  C=0.0
  17: A=0.383  B=0.854  C=0.354
  18: A=0.383  B=0.653  C=0.653
  19: A=0.383  B=0.354  C=0.854
  20: A=0.383  B=0.0  C=0.924
  21: A=0.0  B=1.0  C=0.0
  22: A=0.0  B=0.924  C=0.383
  23: A=0.0  B=0.707  C=0.707
  24: A=0.0  B=0.383  C=0.924
  25: A=0.0  B=0.0  C=1.0
Solving combination 1/1: [1.0,0.0,0.0]
c model file is set to : multi_1.mps
c time limit is set to : 30.00 seconds
c sol path is set to : multi_1.sol
c welcome to Local-MIP solver
c model name: 
c reading mps file takes 0.17 seconds.
c original prob

In [49]:
# Write visualization data to txt file for Python visualizer
viz_file = output_folder * "/" * input_file_name * "_viz_data.txt"
open(viz_file, "w") do io
    println(io, "# Multi-criteria optimization visualization data")
    println(io, "# ideal_point,A,B,C")
    println(io, "# solution,wA,wB,wC,A,B,C,A_norm,B_norm,C_norm")
    println(io, "ideal_point,$(ideal_point[1]),$(ideal_point[2]),$(ideal_point[3])")
    for i in 1:length(solutions)
        wA, wB, wC = solution_weights[i]
        A_val, B_val, C_val = solutions_nonrm[i]
        An, Bn, Cn = solutions[i]
        println(io, "solution,$wA,$wB,$wC,$A_val,$B_val,$C_val,$An,$Bn,$Cn")
    end
end
println("Visualization data written to: $viz_file")

Visualization data written to: results/input_current.txt_viz_data.txt
